# Aksara OCR — publication figures

Generates the paper's figures from the committed result files. **CPU-only, no
GPU or dataset needed** — it just reads the CSVs/JSONs already in the repo. Runs
in a minute; you can run it interactively (it is too quick to bother with a
batch commit).

Figures are shown inline and saved to `/kaggle/working/figures/` as 300-dpi PNG
and PDF. Download them from the **Output** tab (or Save Version to keep them).

In [ ]:
import os
from pathlib import Path

REPO = Path("/kaggle/working/aksantara-ocr")
if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q https://github.com/phoenixfin/aksantara-ocr.git {REPO}
os.chdir(REPO)
!pip install -q matplotlib
print("ready:", Path.cwd())

In [ ]:
import json, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 11,
                 "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})
FIG = Path("/kaggle/working/figures"); FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight")
    print("saved", name)

# All results come from committed files — paths relative to the repo root.
ABL = pd.read_csv("artifacts_kaggle/ablation_results_complete.csv")
ABL["macro_f1_pct"] = ABL["macro_f1"] * 100
print(f"loaded {len(ABL)} ablation runs")

## Figure 1 — Input resolution vs accuracy

In [ ]:
d = ABL[(ABL.augmentation=="medium") & (ABL.pretrained==True)]
g = d.groupby("image_size")["macro_f1_pct"].agg(["mean","std"]).reset_index()

fig, ax = plt.subplots(figsize=(6,4))
ax.errorbar(g.image_size, g["mean"], yerr=g["std"], marker="o", capsize=3, lw=2)
ax.set_xlabel("Input resolution (px)"); ax.set_ylabel("Macro-F1 (%)")
ax.set_title("ResNet-18, unified 895-class task (mean ± std, 3 seeds)")
ax.set_xticks(g.image_size)
for _,r in g.iterrows():
    ax.annotate(f"{r['mean']:.1f}", (r.image_size, r['mean']),
                textcoords="offset points", xytext=(0,8), ha="center", fontsize=9)
save(fig, "fig1_size_ablation"); plt.show()

## Figure 2 — Augmentation strength × pretraining

In [ ]:
d = ABL[ABL.image_size==64]
piv = d.groupby(["augmentation","pretrained"])["macro_f1_pct"].mean().unstack()
order = ["none","light","medium","heavy"]; piv = piv.reindex(order)

fig, ax = plt.subplots(figsize=(6.5,4))
x = np.arange(len(order)); w = 0.38
ax.bar(x-w/2, piv[True],  w, label="Pretrained")
ax.bar(x+w/2, piv[False], w, label="From scratch")
ax.set_xticks(x); ax.set_xticklabels([a.capitalize() for a in order])
ax.set_xlabel("Augmentation strength"); ax.set_ylabel("Macro-F1 (%)")
ax.set_ylim(95, 99); ax.legend()
ax.set_title("ResNet-18 @64px — augmentation barely helps; pretraining does")
save(fig, "fig2_augmentation"); plt.show()

## Figure 3 — Per-script difficulty (layered classifier, end-to-end)

In [ ]:
H = json.load(open("artifacts_colab/results/main/report/hierarchical.json"))
from collections import defaultdict
agg = defaultdict(list)
for seed in H:
    for r in seed["per_script"]:
        agg[r["true_script"]].append(r["end_to_end"]*100)
rows = sorted(((k, np.mean(v)) for k,v in agg.items()), key=lambda x:x[1])
names = [r[0] for r in rows]; vals = [r[1] for r in rows]

fig, ax = plt.subplots(figsize=(6.5,5))
colors = plt.cm.RdYlGn((np.array(vals)-min(vals))/(max(vals)-min(vals)+1e-9))
ax.barh(names, vals, color=colors)
ax.set_xlabel("End-to-end accuracy (%)"); ax.set_xlim(min(vals)-1, 100)
ax.set_title("Per-script end-to-end accuracy (mean, 3 seeds)")
for i,v in enumerate(vals):
    ax.annotate(f"{v:.1f}", (v, i), textcoords="offset points", xytext=(3,0), va="center", fontsize=9)
save(fig, "fig3_per_script"); plt.show()

## Figure 4 — Classical baselines vs deep

In [ ]:
C = pd.read_csv("artifacts_local/results/classical/classical_results.csv")
bars = [(r.classifier.replace("_"," "), r.macro_f1*100) for _,r in C.iterrows()]
# deep reference: layered end-to-end and best unified ablation macro-F1
deep_e2e = np.mean([s["end_to_end_accuracy"]*100 for s in H])
bars += [("deep\n(layered e2e)", deep_e2e)]
bars.sort(key=lambda x:x[1])
labels=[b[0] for b in bars]; vals=[b[1] for b in bars]

fig, ax = plt.subplots(figsize=(6.5,4))
cols = ["#888"]*(len(bars)-1) + ["#2a7"]
ax.bar(labels, vals, color=cols)
ax.set_ylabel("Macro-F1 (%)"); ax.set_ylim(0,100)
ax.set_title("Classical (HOG, 10k train) vs deep — the gap the dataset creates")
for i,v in enumerate(vals):
    ax.annotate(f"{v:.1f}", (i,v), textcoords="offset points", xytext=(0,4), ha="center", fontsize=9)
save(fig, "fig4_classical_vs_deep"); plt.show()

## Figure 5 — Most-confused character pairs
Aggregated across the per-script models (seed 0). These are the genuinely
hard-to-distinguish glyph pairs — the qualitative core of a script paper.

In [ ]:
from collections import Counter
pairs = Counter()
for f in glob.glob("artifacts_colab/results/main/resnet18__per_script__*__s0/result.json"):
    d = json.load(open(f)); script = d["experiment"]["script_filter"]
    for mc in d.get("most_confused", []):
        t = mc["true"].split("/")[-1]; p = mc["predicted"].split("/")[-1]
        pairs[f"{script}: {t}→{p}"] += mc["count"]
top = pairs.most_common(15)[::-1]
labels=[t[0] for t in top]; vals=[t[1] for t in top]

fig, ax = plt.subplots(figsize=(7,5.5))
ax.barh(labels, vals, color="#c55")
ax.set_xlabel("Misclassification count (test set)")
ax.set_title("Top-15 most-confused character pairs")
save(fig, "fig5_confused_pairs"); plt.show()

In [ ]:
print("All figures written to", FIG)
for p in sorted(FIG.glob("*.png")):
    print("  ", p.name)
print("\nDownload from the Output tab, or Save Version to keep them.")